# Thème Numéro 2 - Facteurs Saisonniers et Succès au Speed Dating

## Question 1 - Le succès varie-t-il selon la saison ?
Le succès au speed dating (nombre de matchs obtenus) diffère-t-il entre le printemps et l'automne ?

- **H0** : il n'y a pas de différence de nombre moyen de matchs entre le printemps et l'automne (p > 0.05)
- **H1** : il existe une différence de nombre moyen de matchs entre le printemps et l'automne (p ≤ 0.05)

**Variables analysées :**
- `nb_matchs` : nombre total de matchs par participant (somme de `match == 1` groupée par `iid`)
- `saison` : Spring (vagues 6–9 et 18-21) ou Autumn (vagues 1–5 et 10–17)

Seuil de significativité : α = 0.05

## 0. Chargement des données

In [1]:
import pandas as pd
import numpy as np
from scipy import stats
import plotly.express as px
import plotly.graph_objects as go

In [2]:
df = pd.read_csv("Speed+Dating+Data.csv", encoding="MacRoman")
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8378 entries, 0 to 8377
Columns: 195 entries, iid to amb5_3
dtypes: float64(174), int64(13), object(8)
memory usage: 12.5+ MB


## 1. Création des variables

### Mapping vague -> saison
Le dataset contient 21 vagues (*waves*) d'événements.
D'après la documentation, les vagues 6 à 9 et 18 à 21 se sont tenues au **printemps** (*Spring*), toutes les autres (1–5, 10–17) à l'**automne** (*Autumn*).

On calcule pour chaque individu :
- **nb_matchs** : somme des rencontres où `match == 1` (groupée par `iid`)
- **saison** : Spring ou Autumn, déduite du numéro de vague

In [3]:
SPRING_WAVES = [6, 7, 8, 9, 18, 19, 20, 21]

per_person = (
    df.groupby(['iid', 'wave'])
    .agg(nb_matchs=('match', 'sum'))
    .reset_index()
)

per_person['season'] = per_person['wave'].apply(
    lambda w: 'Spring' if w in SPRING_WAVES else 'Autumn'
)

print(f"Nombre de participants: {len(per_person)}")
print(per_person['season'].value_counts().to_string())
per_person.head()

Nombre de participants: 551
season
Autumn    350
Spring    201


,iid,wave,nb_matchs,season
0,1,1,4,Autumn
1,2,1,2,Autumn
2,3,1,0,Autumn
3,4,1,2,Autumn
4,5,1,2,Autumn


## 2. Statistiques descriptives

In [7]:
descr = per_person.groupby('season')['nb_matchs'].describe().round(2)
print("Statistiques descriptives par saison:\n ")
print(descr)

Statistiques descriptives par saison:
 
        count  mean   std  min  25%  50%  75%   max
season                                             
Autumn  350.0  2.54  2.27  0.0  1.0  2.0  4.0  11.0
Spring  201.0  2.44  2.33  0.0  1.0  2.0  3.0  14.0


## 3. Vérification des conditions du t-test

Avant de réaliser un t-test indépendant, il faut vérifier la **normalité** (test de Shapiro-Wilk sur chaque groupe)


In [8]:
spring_data = per_person[per_person['season'] == 'Spring']['nb_matchs']
autumn_data = per_person[per_person['season'] == 'Autumn']['nb_matchs']

# normalité
stat_s, p_shapiro_s = stats.shapiro(spring_data)
stat_a, p_shapiro_a = stats.shapiro(autumn_data)

print("Test de Shapiro-Wilk (normalité):\n")
print(f"Spring : W = {stat_s:.3f}, p = {p_shapiro_s:.4f}")
print(f"Autumn : W = {stat_a:.3f}, p = {p_shapiro_a:.4f}")

if p_shapiro_s < 0.05 or p_shapiro_a < 0.05:
    print("\nAu moins un groupe ne suit pas une distribution normale (p < 0.05).")
else:
    print("\nLes deux groupes suivent une distribution normale.")



Test de Shapiro-Wilk (normalité):

Spring : W = 0.859, p = 0.0000
Autumn : W = 0.882, p = 0.0000

Au moins un groupe ne suit pas une distribution normale (p < 0.05).


## 4. T-test indépendant : Spring vs Autumn

On compare le nombre moyen de matchs entre les deux saisons via un **t-test indépendant**.
Les deux groupes sont indépendants (participants différents selon la vague).

In [10]:
t_stat, p_ttest = stats.ttest_ind(spring_data, autumn_data)

print("T-Test indépendant:")
print(f"t = {t_stat:.3f}")
print(f"p-value = {p_ttest:.4f}")

if p_ttest < 0.05:
    print("\nH0 rejetée : différence significative du nombre de matchs entre les saisons (p <= 0.05)")
    if spring_data.mean() > autumn_data.mean():
        print(" -> Le printemps génère en moyenne plus de matchs que l'automne.")
    else:
        print("-> L'automne génère en moyenne plus de matchs que le printemps.")
else:
    print("\nH0 non-rejetée : pas de différence significative entre les saisons (p > 0.05)")



T-Test indépendant:
t = -0.519
p-value = 0.6043

H0 non-rejetée : pas de différence significative entre les saisons (p > 0.05)


## 6. Récapitulatif des moyennes

In [11]:
print("Récapitulatif:")

print(f"\nNombre Moyen de matchs (Spring) : {spring_data.mean():.3f} (n = {len(spring_data)})")
print(f"Nombre Moyen de matchs (Autumn) : {autumn_data.mean():.3f} (n = {len(autumn_data)})")
print(f"Différence des moyennes :{abs(spring_data.mean() - autumn_data.mean()):.3f}")

print(f"\nT-test: t = {t_stat:.3f}, p = {p_ttest:.4f}")

Récapitulatif:

Nombre Moyen de matchs (Spring) : 2.438 (n = 201)
Nombre Moyen de matchs (Autumn) : 2.543 (n = 350)
Différence des moyennes :0.105

T-test: t = -0.519, p = 0.6043


## 7. Visualisation

Les graphiques ci-dessous illustrent la distribution du nombre de matchs par saison ainsi que la comparaison des moyennes.

In [13]:
## box plot - distribution par saison
fig_box = px.box(
    per_person,
    x='season',
    y='nb_matchs',
    color='season',
    title="Distribution du nombre de matchs par saison",
    labels={
        'nb_matchs': 'Nombre de Matchs',
        'season': 'Saison'
    },
    color_discrete_map={'Spring': '#2ecc71', 'Autumn': '#e67e22'}
)

fig_box.show()

In [14]:
## histogramme - fréquence du nombre de matchs par saison
fig_hist = px.histogram(
    per_person,
    x='nb_matchs',
    color='season',
    barmode='overlay',
    opacity=0.7,
    nbins=15,
    title="Distribution du nombre de matchs - Spring vs Autumn",
    labels={
        'nb_matchs': 'Nombre de Matchs',
        'count': 'Nombre de participants',
        'season': 'Saison'
    },
    color_discrete_map={'Spring': '#2ecc71', 'Autumn': '#e67e22'}
)

fig_hist.show()

In [15]:
# bar chart : comparaison des moyennes avec barres d'erreur (écart-type)
summary = per_person.groupby('season')['nb_matchs'].agg(['mean', 'std', 'count']).reset_index()
summary['sem'] = summary['std'] / np.sqrt(summary['count'])  # erreur standard

fig_bar = px.bar(
    summary,
    x='season',
    y='mean',
    error_y='sem',
    color='season',
    text=summary['mean'].round(2),
    title="Nombre moyen de matchs par saison (± erreur standard)",
    labels={
        'mean': 'Nombre moyen de matchs',
        'season': 'Saison'
    },
    color_discrete_map={'Spring': '#2ecc71', 'Autumn': '#e67e22'}
)
fig_bar.update_traces(textposition='outside')
fig_bar.show()

## Conclusion

Le t-test indépendant ne permet pas de rejeter H0:
- **T-test indépendant** : t = -0.519, p = 0.604

**Il n'existe pas de différence significative du nombre de matchs entre le printemps et l'automne.**

Les moyennes sont très proches:
- **Spring**: 2.438 matchs en moyenne (n = 201)
- **Autumn** : 2.543 matchs en moyenne (n = 350)
- **Différence** : 0.105 match seulement

Comme le confirment visuellement les trois graphiques, les distributions par saison se superposent presque parfaitement : même médiane (2 matchs), même dispersion interquartile, et des barres de moyennes quasi à la même hauteur.
L'automne affiche une moyenne très légèrement supérieure, mais cet écart est sans significance statistique.

*A noter: la normalité n'est vérifiée dans aucun des deux groupes (Shapiro-Wilk p < 0.001 pour Spring et Autumn), ce qui est attendu pour une variable de comptage à distribution asymétrique. Le t-test reste néanmoins robuste pour des échantillons de cette taille (n > 30).*

### Ce que ça nous dit
Le contexte saisonnier ne semble pas influencer le nombre de matchs lors d'un speed dating. 
Le nombre de matchs obtenus paraît indépendant de la période de l'année où l'événement se déroule, ce qui suggère que d'autres facteurs pourraient jouer un rôle plus déterminant que la saison.

### Limites à considérer
- Les distributions du nombre de matchs n'est pas normale dans les deux groupes (Shapiro-Wilk p < 0.001).
- Les effectifs sont désiquilibrés (n = 201 Spring vs n = 350 Autumn), ce qui réduit légèrement la puissance statistique côté Spring